In [ ]:
# Import pytorch
import torch
from torch import nn
import random
# Import torch vision
import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# Import matplotlib
import matplotlib.pyplot as plt

# Check versions
print(torch.__version__, torchvision.__version__)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

from torch.utils.data import Dataset
from torchvision import transforms
from typing import Tuple, Dict, List # for typehints

In [ ]:
def find_classes(directory: str) -> Tuple[List[str], Dict[str, int]]:
  """Finds the class folder names in a target directory."""
  classes = sorted([entry.name for entry in list(os.scandir(directory)) if entry.is_dir()])

  if not classes:
    raise FileNotFoundError(f"Couldn't find any classes in directory... Please check file structure")

  class_to_idx = {class_name: i for i, class_name in enumerate(classes)}

  return classes, class_to_idx

In [ ]:
find_classes(Path("../new_data/UAP_Detector_Dataset\max_pool_layer_1/train"))

In [ ]:
ROOT_DIR = ""
train_dir = Path(f"{ROOT_DIR}/train")
test_dir = Path(f"{ROOT_DIR}/test")

In [ ]:
from torch.utils.data import Dataset
from torch import Tensor

class FeatureMapFolder(Dataset):
  def __init__(self, targ_dir: str, transform=None):
    super().__init__()
    self.paths = list(Path(targ_dir).glob("*/*.pt"))
    self.transform = transform
    self.classes, self.class_to_idx = find_classes(targ_dir)
    
  def load_feature_map(self, index: int) -> Tensor:
    "Opens an image via a path and returns it"
    image_path = self.paths[index]
    return torch.load(image_path)
  
  def aggregate_channels(self, feature_map):    
    # feature_map: [channels, height, width] for one sample
    return torch.sum(feature_map, 0) / feature_map.shape[0]

  def __len__(self):
    "Returns the total number of samples"
    return len(self.paths)


  def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
    "Returns one sample of data, data and label (X, y)"
    img = self.load_feature_map(index)
    img = img.squeeze()
    img = self.aggregate_channels(img)  
    img = img.unsqueeze(0) 
    class_name = self.paths[index].parent.name 

    class_idx = self.class_to_idx[class_name]


    # Transform if necessary
    if self.transform:
      return self.transform(img), class_idx
    else:
      return img, class_idx

In [ ]:
from torchvision.transforms import transforms

train_transforms = transforms.Compose([
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.ToTensor()
])

In [ ]:
train_data = FeatureMapFolder(train_dir)
test_data = FeatureMapFolder(test_dir)

In [ ]:
def random_from_iter(iterable):
    random_item = None
    for i, item in enumerate(iterable, start=1):
        if random.randint(1, i) == 1:
            random_item = item
    return random_item

def test_dataset():
    layer, label = random_from_iter(iter(train_data))
    print(layer.shape, label)
    plt.imshow(layer.squeeze(0).cpu())
test_dataset()

In [ ]:
import random

def random_k_from_iter(iterator, k):
    reservoir = []
    for i, value in enumerate(iterator):
        if i < k:
            reservoir.append(value)
        else:
            j = random.randint(0, i)
            if j < k:
                reservoir[j] = value
    return reservoir

def display_random_images(dataset: torch.utils.data.Dataset,
              classes: List[str],
              n:int = 10,
              display_shape: bool = True,
              seed:int = None):
  # 2. Adjust display if n is too high
  if n > 10:
    n=10
    print("For display purposes, n should not be larger than 10. Setting n=10 and removing shape display")

  # 3. Set the seed
  if seed:
    random.seed(seed)
  
  # 4. Get random sample indexes
  random_samples_idx = random_k_from_iter(dataset, n)
  # Take n random indices within the length of dataset

  # 5. Setup plot
  plt.figure(figsize=(16, 8))

  # 6. Loop through random indices and plot them with matplotlib
  for i, targ_sample in enumerate(random_samples_idx):
    print(targ_sample)
    targ_image, targ_label = targ_sample


    # Plot adjusted samples
    plt.subplot(1, n, i+1)
    plt.imshow(targ_image.squeeze(0).cpu())
    plt.axis(False)
    if classes:
      title = f"Class: {classes[targ_label]}"
    plt.title(title)

In [ ]:
# Display random images from the FeatureFolder created dataset
display_random_images(
    dataset=train_data,
    n=5,
    classes=train_data.classes
)

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE=16

train_dataloader = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_dataloader = DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

img, label = next(iter(train_dataloader))

img.shape, label.shape # (torch.Size([16, 1, 32, 32]), torch.Size([16]))

In [ ]:
def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    device=device
  ):
  model.train()
  
  train_loss, train_acc = 0, 0
  for batch, (X, y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)

    y_pred = model(X)
    y_pred = y_pred.squeeze()
    loss = loss_fn(y_pred, y.float())
    train_loss += loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    y_pred_class = torch.round(torch.sigmoid(y_pred))
    train_acc += (y_pred_class == y).sum().item() / len(y_pred)

  # Adjust metrics to get the average loss and acc per batch
  train_loss = train_loss / len(dataloader) 
  # len of the dataloader is number of batches
  train_acc = train_acc / len(dataloader)

  return train_loss, train_acc


In [ ]:
def test_step(
    model=torch.nn.Module,
    dataloader=torch.utils.data.DataLoader,
    loss_fn=torch.nn.Module,
    device=device):
  model.eval()

  test_loss, test_acc = 0, 0
  with torch.inference_mode():
    for batch, (X, y) in enumerate(dataloader):
      X, y = X.to(device), y.to(device)

      test_pred = model(X)
      test_pred = test_pred.squeeze()
      loss = loss_fn(test_pred, y.float())
      test_loss += loss.item()
      
      test_acc += (torch.round(torch.sigmoid(test_pred))==y).sum().item() / len(test_pred)

    test_loss /= len(dataloader)
    test_acc /= len(dataloader)

  return test_loss, test_acc
 

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.best_loss = None
        self.no_improvement_count = 0
        self.stop_training = False
    
    def check_early_stop(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.no_improvement_count = 0
        else:
            self.no_improvement_count += 1
            if self.no_improvement_count >= self.patience:
                self.stop_training = True
                if self.verbose:
                    print("Stopping early as no improvement has been observed.")

In [ ]:
from tqdm.auto import tqdm

def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 5,
    device=device, saveBestModel = True, model_name="model_newest"):
  
  results = {
      "train_loss": [],
      "train_acc": [],
      "test_loss": [],
      "test_acc": [],
      "best_epoch": 0,
      "best_test_loss": float('inf'),
      "best_test_acc": 0.0,
  }
  if (saveBestModel):
    best_val_loss = float('inf')

  early_stopping = EarlyStopping(delta=0.001, verbose=True)
  for epoch in tqdm(range(epochs)):
    train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
    test_loss, test_acc = test_step(model, test_dataloader, loss_fn, device)
    
    print(f"Epoch: {epoch}\n----")
    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")
    print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

  
    early_stopping.check_early_stop(test_loss)

    
    # Update results directory
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["test_loss"].append(test_loss)
    results["test_acc"].append(test_acc)

    if (early_stopping.stop_training):
        print(f"Terminated early at loss {test_loss:.4f}")             
        break
    if (test_loss < best_val_loss and saveBestModel):
      best_val_loss = test_loss
      results["best_epoch"] = epoch
      results["best_test_loss"] = test_loss
      results["best_test_acc"] = test_acc
      # Save  model checkpoint
      torch.save(model.state_dict(), f"./models_aggregate/{model_name}_best.pth")
      print(f"Saved new best model at epoch {epoch} with loss {test_loss:.4f}")

  return results

In [ ]:
class Model_1(nn.Module):
    """Baseline model for classification of adversarial samples"""
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_shape, hidden_units),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_units, output_shape),
            
        )
    def forward(self, x):
        return self.layer_stack(x)


In [ ]:
model_1 = Model_1(input_shape=32*32, hidden_units=10, output_shape=1)
model_1 = model_1.to(device)

In [ ]:

from torchinfo import summary
summary(
    model=model_1,
    input_size=[BATCH_SIZE, 32*32]
)

In [ ]:
NUM_EPOCHS = 50

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.0001)


from timeit import default_timer as timer
start_time = timer()
torch.manual_seed(42)
# Train model_0
model_0_training_results = train(model=model_1,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device, saveBestModel=True, model_name="model_1")

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

In [ ]:
def plot_loss_curves(results: Dict[str, List[float]]):
  """Plots training curves of a results dictionary"""
  # Get the loss values of the results dictionary 
  loss = results["train_loss"]
  test_loss = results["test_loss"]

  # Get the acc values of the results dictionary 
  accuracy = results["train_acc"]
  test_accuracy = results["test_acc"]
  # Figure out how many epochs there were
  # no. of epochs is same as length of the train_loss array
  # as we appended the loss per epoch
  epochs = range(len(results["train_loss"]))

  # Setup a plot
  plt.figure(figsize=(15, 7))

  # Plot the loss
  plt.subplot(1, 2, 1)
  plt.plot(epochs, loss, label="train_loss")
  plt.plot(epochs, test_loss, label="test_loss")
  plt.title("Loss")
  plt.xlabel("Epochs")
  plt.legend()

  # Plot the accuracy
  plt.subplot(1, 2, 2)
  plt.plot(epochs, accuracy, label="train_accuracy")
  plt.plot(epochs, test_accuracy, label="test_accuracy")
  plt.title("Accuracy")
  plt.xlabel("Epochs")
  plt.legend();

In [ ]:
plot_loss_curves(model_0_training_results)

In [ ]:
import json
def saveModelAndResults(model: torch.nn.Module, results: Dict[str, List[float]], model_name: str):
    """Saves a model and its training results to disk"""
    results_save_path = f"./results_aggregate/{model_name}_training_results.json"
    model_save_path = f"./models_aggregate/{model_name}.pth"
    if os.path.exists(results_save_path):
        print("Results file already exists. Terminating...")
        return
    if os.path.exists(model_save_path):
        print("Model file already exists. Terminating...")
        return
    # Save the model state dict
    torch.save(obj=model.state_dict(), f=model_save_path)
    # Save the results dictionary
   
    with open(results_save_path, "w") as f:
        json.dump(results, f)
    print(f"Saved {model_name}!")

In [ ]:
saveModelAndResults(model_0, model_0_training_results, "model_0")

In [ ]:
BATCH_SIZE = 128

In [ ]:
class Model_2(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            # Create a conv layer
            nn.Conv2d(in_channels=input_shape,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1), # hyperparameters
            nn.ReLU(),
            nn.BatchNorm2d(hidden_units),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout(0.5),
        )
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2560, hidden_units),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(hidden_units, output_shape),
        )
    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.layer_stack(x)
        return x

In [ ]:
model_2 = Model_2(input_shape=1, hidden_units=10, output_shape=1)
model_2 = model_2.to(device)

In [ ]:

from torchinfo import summary
summary(
    model=model_2,
    input_size=[BATCH_SIZE, 1, 32, 32]
)

In [ ]:
NUM_EPOCHS = 80

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(params=model_2.parameters(), lr= 5e-5, weight_decay=5e-2)


from timeit import default_timer as timer
start_time = timer()
torch.manual_seed(42)
torch.cuda.manual_seed(42)
model_2_training_results = train(model=model_2,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device, saveBestModel=True, model_name="model_2")

end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

In [ ]:
plot_loss_curves(model_2_training_results)

In [ ]:
saveModelAndResults(model_2, model_2_training_results, "model_2")